In [14]:
# Make sure to import packages and run code from the initial_setup notebook.

# The below code will run Bayesian Optimization in PyTorch, or BoTorch.

# Generating default random value for consistency.
torch.manual_seed(19)

# Some basic setup recommended in the documentation.
dtype = torch.double

# In this example, we'll use Function 1. To run this analysis on a different function, simply swap it out.
# For example, insert df_function_5 instead of df_function_1.
data = df_function_1

# To use this code for other functions, add X values. For example, for Function 5, the X values would be:
# X_1 = df_cv['X_1'].values
# X_2 = df_cv['X_2'].values
# X_3 = df_cv['X_3'].values
# X_4 = df_cv['X_4'].values
X_1 = data['X_1'].values
X_2 = data['X_2'].values

# y stays the same for all functions.
y = data['y'].values

# Aggressive output scaling plus StandardScaler for Function 1. For Functions 2-8, I used the yeo-johnson power transformation below as recommended by HEBO. 
# When running Functions 2-8, it's recommended to comment out this aggressive scaling as it will negatively impact your results.
alpha = 0.02
y_eng = np.sign(y) * np.power(np.abs(y), alpha)

# Scale y values for Functions 2-8. 
# pt = PowerTransformer(method='yeo-johnson')
# y_eng = pt.fit_transform(y.reshape(-1, 1))

# Add additional X values if running other functions, similar to the above. For example, running this on Function 5
# would look like points = np.column_stack((X_1, X_2, X_3, X_4))
points = np.column_stack((X_1, X_2))

# Define bounds as a PyTorch tensor. The number of bounds should equal the number of dimensions in the function. For example,
# for Function 5, it would look like this:
# bounds = torch.tensor([
#     [0.0, 0.0, 0.0, 0.0], 
#     [0.999999, 0.999999, 0.999999, 0.999999]
# ], dtype=dtype)
bounds = torch.tensor([
    [0.0, 0.0], 
    [0.999999, 0.999999]
], dtype=dtype)

# Convert X and y to tensors.
train_X = torch.tensor(points, dtype=dtype)
train_y = torch.tensor(y_eng, dtype=dtype).unsqueeze(-1) # Comment this out for Functions 2-8.
# For Functions 2-8, use the below line for train_y, dropping the .unsqueeze(-1).
# train_y = torch.tensor(y_eng, dtype=dtype) 
best_value = train_y.max().item()

# Fit the model.
model = SingleTaskGP(
    train_X, 
    train_y,
    input_transform=Normalize(d=train_X.shape[-1], bounds=bounds), # BoTorch normalizes inputs.
    # Comment out the below line if you're using this on Functions 2-8 as it will get screwed up by the yeo-johnson transform.
    # outcome_transform=Standardize(m=train_y.shape[-1]) # BoTorch standardizes outputs.
)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

# Print length scales and the noise estimate.
print(f'Lengthscales: {model.covar_module.lengthscale}')
print(f'Noise estimate: {model.likelihood.noise}')

# Setup the acquisition function.
# For closed form functions, LogExpectedImprovement is generally recommended
# over sampling for q=1, which we're using for this project.
acq_func = LogExpectedImprovement(
    model=model,
    best_f=best_value,
    maximize=True
)

# Optimize the model.
candidate, acq_value = optimize_acqf(
    acq_function=acq_func,
    bounds=bounds,
    q=1,
    num_restarts=200, 
    raw_samples=10000 # Even though this looks scary, this runs very quickly. 
)

# Output recommendation. For higher-dimensional functions, add X values as outlined previously.
rec_X1 = candidate[0, 0].item()
rec_X2 = candidate[0, 1].item()

# Add X values for higher-dimensional functions as outlined previously.
print(f"Recommended X_1: {rec_X1:.6f}")
print(f"Recommended X_2: {rec_X2:.6f}")

# Return expected output.
posterior = model.posterior(candidate)

# Inverse transform for Function 1.
pred_y_eng = posterior.mean.item()
pred_y_orig = np.sign(pred_y_eng) * np.power(np.abs(pred_y_eng), 1 / alpha)

# Inverse transform for Functions 2-8.
# pred_y_orig = pt.inverse_transform([[pred_y_eng]])[0, 0]

print(f"Expected outcome (original scale): {pred_y_orig}")

Lengthscales: Parameter containing:
tensor([[0.1549, 0.0269]], dtype=torch.float64, requires_grad=True)
Noise estimate: Parameter containing:
tensor([0.0042], dtype=torch.float64, requires_grad=True)
Recommended X_1: 0.442150
Recommended X_2: 0.355588
Expected outcome (original scale): 0.0003215720970786468
